# Arquitectura de una Red Neuronal Profunda (Deep Learning) y Redes Convolucionales (CNN)

**Autor:** Diego Yanzir Gutiérrez Camelo
**Institución:** Especialización en Inteligencia Artificial, Universidad de Cundinamarca
**Curso:** 604010 – Inteligencia Artificial: Conceptos
**Actividad:** R2-A4-S10 Gráfica – Red neuronal (Deep Learning)
**Gestor del conocimiento:** Paul Alexander Diaz Montaña
**Fecha:** 27 de octubre de 2026

---

Este cuaderno replica en Python la gráfica del póster y la lleva a la práctica:

1. **Figura 1** – dibujo de la red profunda de 5 capas (4-6-5-6-3) con pesos *w*, sesgos *b*, ReLU y Softmax.
2. **Figura 2** – etapa convolucional calculada paso a paso con NumPy (imagen 6×6 de una “X”, dos filtros diagonales, ReLU y *max pooling*), siguiendo a Serrano.Academy (2017a).
3. **Modelo real en Keras** – CNN con 3 capas ocultas (Convolución → Pooling → Densa) entrenada con **aprendizaje supervisado** para clasificar las letras X, O y +.
4. **Figura 3** – la misma gráfica de la red, ahora con las probabilidades que predice el modelo entrenado.

> En Colab: *Entorno de ejecución → Ejecutar todas*. No necesita GPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Rectangle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams['font.family'] = 'DejaVu Sans'
print('TensorFlow', tf.__version__)

## 1. Figura 1 – Arquitectura de la red neuronal profunda

La función `dibujar_red()` reproduce la figura central del póster:

| Capa | Rol | Neuronas | Activación |
|---|---|---|---|
| Entrada (*Input Layer*) | Recibe los datos crudos (píxeles) | 4 | — |
| Oculta 1 – Convolución | Extracción de características | 6 | ReLU |
| Oculta 2 – Pooling | Reducción espacial | 5 | Max-Pooling 2×2 |
| Oculta 3 – Densa (*Fully Connected*) | Combinación global | 6 | ReLU |
| Salida (*Output Layer*) | Clasificación en 3 clases | 3 | Softmax |

Cada línea es una conexión sináptica con su peso *w*; los cuadros **+1** son los sesgos *b*.

In [ ]:
CAPAS = [
    dict(n=4, c='#2563eb', d='#1e3a8a', t1='CAPA DE ENTRADA', t2='Input Layer',             t3='Datos crudos (píxeles)',        lab='x', act='Sin activación'),
    dict(n=6, c='#0d9488', d='#134e4a', t1='CAPA OCULTA 1',   t2='Convolución',             t3='Extracción de características', lab='c', act='ReLU'),
    dict(n=5, c='#d97706', d='#78350f', t1='CAPA OCULTA 2',   t2='Agrupamiento (Pooling)',  t3='Reducción espacial',            lab='p', act='Max-Pooling 2×2'),
    dict(n=6, c='#7c3aed', d='#4c1d95', t1='CAPA OCULTA 3',   t2='Densa (Fully Connected)', t3='Combinación global',            lab='d', act='ReLU'),
    dict(n=3, c='#e11d48', d='#881337', t1='CAPA DE SALIDA',  t2='Output Layer',            t3='Clasificación (3 clases)',      lab=r'\hat{y}', act='Softmax'),
]

def dibujar_red(clases=('Gato', 'Perro', 'Ave'), probs=(0.78, 0.15, 0.07),
                pixeles=(0.95, 0.40, 0.70, 0.10), titulo=None, archivo=None):
    # Coordenadas en un lienzo de 1100 × 760 (y crece hacia abajo, como en el póster)
    W, H, R, CY, GAP = 1100, 760, 21, 330, 70
    xs = [190, 390, 580, 770, 950]
    for L, x in zip(CAPAS, xs):
        L['x'] = x
        L['ys'] = [CY + (i - (L['n'] - 1) / 2) * GAP for i in range(L['n'])]

    fig, ax = plt.subplots(figsize=(16, 16 * H / W), dpi=110)
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect('equal'); ax.axis('off')
    fig.patch.set_facecolor('white')
    T = dict(ha='center', va='center')

    # Fondo y títulos de cada capa
    for L in CAPAS:
        ax.add_patch(FancyBboxPatch((L['x'] - 56, 84), 112, 486, boxstyle='round,pad=0,rounding_size=14',
                                    fc=L['c'], ec=L['c'], alpha=0.08, lw=1.2, zorder=0))
        ax.add_patch(FancyBboxPatch((L['x'] - 56, 84), 112, 486, boxstyle='round,pad=0,rounding_size=14',
                                    fc='none', ec=L['c'], alpha=0.4, lw=1.2, zorder=0))
        ax.text(L['x'], 24, L['t1'], fontsize=10.5, weight='bold', color=L['d'], **T)
        ax.text(L['x'], 45, L['t2'], fontsize=10, weight='semibold', color=L['c'], **T)
        ax.text(L['x'], 64, L['t3'], fontsize=8.5, style='italic', color='#475569', **T)

    # Imagen de entrada 2×2: cada píxel alimenta una neurona de entrada
    cell, gx, gy = 34, 30, CY - 34
    for i, v in enumerate(pixeles):
        px, py = gx + (i % 2) * cell, gy + (i // 2) * cell
        ax.plot([px + cell / 2, CAPAS[0]['x'] - R], [py + cell / 2, CAPAS[0]['ys'][i]],
                ls=':', color='#64748b', lw=1, zorder=1)
        ax.add_patch(Rectangle((px, py), cell, cell, fc=str(1 - v), ec='white', lw=1.5, zorder=2))
    ax.add_patch(Rectangle((gx, gy), 2 * cell, 2 * cell, fc='none', ec='#334155', lw=1.5, zorder=3))
    ax.text(gx + cell, gy + 2 * cell + 18, 'Imagen 2×2', fontsize=9, weight='semibold', color='#334155', **T)
    ax.text(gx + cell, gy + 2 * cell + 34, '(4 píxeles)', fontsize=8, color='#64748b', **T)

    # Conexiones sinápticas entre todas las neuronas de capas contiguas
    for A, B in zip(CAPAS[:-1], CAPAS[1:]):
        for y1 in A['ys']:
            for y2 in B['ys']:
                ax.plot([A['x'] + R, B['x'] - R], [y1, y2], color=A['c'], alpha=0.32, lw=0.9, zorder=1)

    # Sesgos b: unidad +1 hacia la capa siguiente (la capa de pooling no tiene sesgo)
    for fr, to in [(0, 1), (2, 3), (3, 4)]:
        A, B, by = CAPAS[fr], CAPAS[to], 104
        for y2 in B['ys']:
            ax.plot([A['x'] + 9, B['x'] - R], [by, y2], ls=(0, (4, 3)), color='#64748b', alpha=0.6, lw=0.9, zorder=1)
        ax.add_patch(FancyBboxPatch((A['x'] - 11, by - 11), 22, 22, boxstyle='round,pad=0,rounding_size=4',
                                    fc='white', ec='#334155', lw=1.4, zorder=4))
        ax.text(A['x'], by, '+1', fontsize=7.5, weight='bold', color='#334155', zorder=5, **T)
        ax.text(A['x'] - 16, by, rf'$b^{{({to})}}$', fontsize=11, color='#334155', ha='right', va='center')

    # Pesos destacados w
    def peso(la, i, j, etiqueta, dy):
        A, B = CAPAS[la], CAPAS[la + 1]
        x1, y1, x2, y2 = A['x'] + R, A['ys'][i], B['x'] - R, B['ys'][j]
        ax.plot([x1, x2], [y1, y2], color='#0f172a', lw=2.4, zorder=3)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + dy, etiqueta, fontsize=11, color='#0f172a', zorder=6,
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='#0f172a', lw=1), **T)
    peso(0, 0, 0, r'$w_{11}^{(1)}$', -18)
    peso(3, 1, 0, r'$w_{21}^{(4)}$', -16)

    # Neuronas
    for L in CAPAS:
        for i, y in enumerate(L['ys']):
            ax.add_patch(Circle((L['x'], y), R + 1.5, fc='none', ec=L['d'], lw=1.2, zorder=7))
            ax.add_patch(Circle((L['x'], y), R, fc=L['c'], ec='white', lw=2.5, zorder=8))
            ax.text(L['x'], y, rf"${L['lab']}_{{{i + 1}}}$", fontsize=11, color='white', zorder=9, **T)

    # Probabilidades Softmax en la salida
    out = CAPAS[4]
    for i, (nombre, p) in enumerate(zip(clases, probs)):
        x, y = out['x'] + R + 10, out['ys'][i]
        ax.text(x, y - 6, f'{nombre} · {p:.2f}'.replace('.', ','), fontsize=9.5, weight='bold', color='#881337', va='center')
        ax.add_patch(Rectangle((x, y + 5), 90, 6, fc='#fecdd3', ec='none'))
        ax.add_patch(Rectangle((x, y + 5), 90 * p, 6, fc='#e11d48', ec='none'))
    ax.text(out['x'] + R + 10, out['ys'][2] + 34, f'Σ = {sum(probs):.2f}'.replace('.', ','),
            fontsize=8.5, style='italic', color='#475569', va='center')

    # Insignias de activación
    for L in CAPAS:
        w = len(L['act']) * 7.4 + 22
        ax.add_patch(FancyBboxPatch((L['x'] - w / 2, 581), w, 26, boxstyle='round,pad=0,rounding_size=13',
                                    fc=L['c'], ec='none'))
        ax.text(L['x'], 594, L['act'], fontsize=9, weight='bold', color='white', **T)

    # Propagación hacia adelante y retropropagación
    ax.text(575, 628, r'Propagación hacia adelante (forward pass):  $z = \sum_i w_i x_i + b \;\rightarrow\; a = f(z)$',
            fontsize=10, color='#334155', **T)
    ax.annotate('', xy=(1000, 645), xytext=(150, 645), arrowprops=dict(arrowstyle='-|>', color='#334155', lw=1.8))
    ax.annotate('', xy=(150, 662), xytext=(1000, 662),
                arrowprops=dict(arrowstyle='-|>', color='#be123c', lw=1.8, ls='--'))
    ax.text(575, 682, r'Retropropagación del error (backpropagation):  $w \leftarrow w - \eta\,\partial L/\partial w$',
            fontsize=10, color='#be123c', **T)

    # Leyenda
    ax.add_patch(FancyBboxPatch((10, 702), 1080, 50, boxstyle='round,pad=0,rounding_size=10', fc='#f8fafc', ec='#cbd5e1'))
    ax.text(26, 727, 'Leyenda:', fontsize=9.5, weight='bold', va='center')
    ax.plot([102, 140], [727, 727], color='#0f172a', lw=2.4)
    ax.text(148, 727, 'Conexión sináptica = peso w', fontsize=9, va='center')
    ax.add_patch(FancyBboxPatch((350, 716), 22, 22, boxstyle='round,pad=0,rounding_size=4', fc='white', ec='#334155', lw=1.4))
    ax.text(361, 727, '+1', fontsize=7.5, weight='bold', color='#334155', **T)
    ax.plot([375, 405], [727, 727], ls=(0, (4, 3)), color='#64748b', lw=1.3)
    ax.text(412, 727, 'Sesgo b (desplaza el umbral)', fontsize=9, va='center')
    ax.add_patch(FancyBboxPatch((618, 715), 56, 24, boxstyle='round,pad=0,rounding_size=12', fc='#0d9488', ec='none'))
    ax.text(646, 727, 'ReLU', fontsize=8.5, weight='bold', color='white', **T)
    ax.text(681, 727, r'$f(z) = \max(0, z)$', fontsize=9.5, va='center')
    ax.add_patch(FancyBboxPatch((800, 715), 76, 24, boxstyle='round,pad=0,rounding_size=12', fc='#e11d48', ec='none'))
    ax.text(838, 727, 'Softmax', fontsize=8.5, weight='bold', color='white', **T)
    ax.text(883, 727, r'$\sigma(z)_i = e^{z_i} / \sum_j e^{z_j}$', fontsize=9.5, va='center')

    if titulo:
        fig.suptitle(titulo, fontsize=13, weight='bold', x=0.06, ha='left', y=0.99)
    plt.tight_layout()
    if archivo:
        fig.savefig(archivo, dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

dibujar_red(titulo='Figura 1. Red neuronal profunda de cinco capas (4-6-5-6-3)',
            archivo='figura1_red_profunda.png')

## 2. Figura 2 – Etapa convolucional paso a paso

Como en el video de Serrano.Academy (2017a): una imagen de 6×6 píxeles con la letra **X** (1 = píxel oscuro, −1 = píxel claro) se recorre con dos filtros de 3×3 que buscan diagonales. Cada resultado se divide entre 9, así que **1,00 significa que el patrón coincide por completo**. Después ReLU elimina los negativos y el *max pooling* 2×2 conserva el valor más alto de cada zona.

In [ ]:
X_img = np.array([
    [ 1, -1, -1, -1, -1,  1],
    [-1,  1, -1, -1,  1, -1],
    [-1, -1,  1,  1, -1, -1],
    [-1, -1,  1,  1, -1, -1],
    [-1,  1, -1, -1,  1, -1],
    [ 1, -1, -1, -1, -1,  1]], dtype=float)

filtro_1 = np.array([[ 1, -1, -1], [-1,  1, -1], [-1, -1,  1]], dtype=float)   # diagonal \
filtro_2 = np.array([[-1, -1,  1], [-1,  1, -1], [ 1, -1, -1]], dtype=float)   # diagonal /

def convolucion(img, k):
    # Convolución 'valid' (sin relleno), normalizada entre el número de píxeles del filtro
    h, w = img.shape[0] - k.shape[0] + 1, img.shape[1] - k.shape[1] + 1
    return np.array([[np.sum(img[i:i + 3, j:j + 3] * k) / k.size for j in range(w)] for i in range(h)])

relu = lambda z: np.maximum(0, z)

def max_pooling(m, t=2):
    return m.reshape(m.shape[0] // t, t, m.shape[1] // t, t).max(axis=(1, 3))

mapa_1, mapa_2 = convolucion(X_img, filtro_1), convolucion(X_img, filtro_2)
pool_1, pool_2 = max_pooling(relu(mapa_1)), max_pooling(relu(mapa_2))
aplanado = np.concatenate([pool_1.ravel(), pool_2.ravel()])

np.set_printoptions(precision=2, suppress=True)
print('Mapa de características 1 (4×4):\n', mapa_1)
print('\nMax pooling 1 (2×2):\n', pool_1)
print('\nVector aplanado (8 valores que entran a la red densa):\n', aplanado)

In [ ]:
def heatmap(ax, m, titulo, cmap):
    ax.imshow(m, cmap=cmap, vmin=-1, vmax=1)
    for (i, j), v in np.ndenumerate(m):
        etiqueta = f'{v:.2f}'.replace('.', ',') if np.any(m % 1) else f'{int(v)}'
        ax.text(j, i, etiqueta.replace('-', '−'), ha='center', va='center', fontsize=9,
                color='white' if v > 0.5 else 'black')   # texto claro solo sobre celdas oscuras
    ax.set_title(titulo, fontsize=10, weight='bold'); ax.set_xticks([]); ax.set_yticks([])

fig, axs = plt.subplots(2, 5, figsize=(17, 9), gridspec_kw=dict(width_ratios=[1.5, 0.9, 1.2, 0.8, 0.5], hspace=0.35))
for fila, (k, mapa, pool, nombre, cm) in enumerate([(filtro_1, mapa_1, pool_1, 'Filtro 1 (diagonal \\)', 'Blues'),
                                                    (filtro_2, mapa_2, pool_2, 'Filtro 2 (diagonal /)', 'Purples')]):
    heatmap(axs[fila, 0], X_img, 'Imagen de entrada 6×6 (letra X)', 'Greys')
    heatmap(axs[fila, 1], k, nombre + '\n3×3', 'Oranges')
    heatmap(axs[fila, 2], mapa, 'Convolución → mapa 4×4', cm)
    heatmap(axs[fila, 3], pool, 'ReLU + Max pooling 2×2', cm)
heatmap(axs[0, 4], aplanado.reshape(-1, 1), 'Aplanado\n(8 valores)', 'Greens')
axs[1, 4].axis('off')
axs[1, 4].text(0.5, 0.5, 'Entrada de la\nred densa\n8 → … → 3', ha='center', va='center', fontsize=11,
               weight='bold', color='#7c3aed', transform=axs[1, 4].transAxes)
fig.suptitle('Figura 2. Etapa convolucional que extrae las características que recibe la red',
             fontsize=13, weight='bold', x=0.02, ha='left')
fig.savefig('figura2_etapa_convolucional.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

## 3. Aprendizaje supervisado: construir y entrenar la CNN en Keras

**Datos etiquetados.** Se generan imágenes de 6×6 de tres clases (**X**, **O**, **+**) con ruido aleatorio y desplazamientos, cada una con su etiqueta correcta.

**Modelo.** Se amplía la red a **3 capas ocultas**, igual que en la Figura 1:

`Entrada (6×6×1) → Conv2D (ReLU) → MaxPooling2D → Flatten → Dense (ReLU) → Dense (3, Softmax)`

In [ ]:
CLASES = ['X', 'O', '+']
plantillas = {
    'X': X_img,
    'O': np.array([[-1, -1, -1, -1, -1, -1],
                   [-1,  1,  1,  1,  1, -1],
                   [-1,  1, -1, -1,  1, -1],
                   [-1,  1, -1, -1,  1, -1],
                   [-1,  1,  1,  1,  1, -1],
                   [-1, -1, -1, -1, -1, -1]], dtype=float),
    '+': np.array([[-1, -1,  1,  1, -1, -1],
                   [-1, -1,  1,  1, -1, -1],
                   [ 1,  1,  1,  1,  1,  1],
                   [ 1,  1,  1,  1,  1,  1],
                   [-1, -1,  1,  1, -1, -1],
                   [-1, -1,  1,  1, -1, -1]], dtype=float),
}

def generar_datos(n_por_clase=400, ruido=0.25):
    X, y = [], []
    for etiqueta, nombre in enumerate(CLASES):
        for _ in range(n_por_clase):
            img = np.roll(plantillas[nombre], shift=np.random.randint(-1, 2), axis=np.random.randint(0, 2))
            img = img + np.random.normal(0, ruido, img.shape)               # ruido
            img[np.random.rand(*img.shape) < 0.05] *= -1                    # píxeles invertidos
            X.append(img); y.append(etiqueta)
    X = np.array(X)[..., np.newaxis].astype('float32')
    y = np.array(y)
    idx = np.random.permutation(len(y))
    return X[idx], y[idx]

X_train, y_train = generar_datos(400)
X_test, y_test = generar_datos(100)
print('Entrenamiento:', X_train.shape, ' Prueba:', X_test.shape)

fig, axs = plt.subplots(1, 9, figsize=(14, 2))
for ax, img, et in zip(axs, X_train[:9], y_train[:9]):
    ax.imshow(img[..., 0], cmap='Greys', vmin=-1.5, vmax=1.5); ax.set_title(f'Etiqueta: {CLASES[et]}', fontsize=9); ax.axis('off')
plt.suptitle('Ejemplos de datos etiquetados (aprendizaje supervisado)', fontsize=11, weight='bold')
plt.show()

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(6, 6, 1), name='entrada'),
    layers.Conv2D(6, (3, 3), activation='relu', name='oculta1_convolucion'),   # 6 filtros 3×3 → 4×4×6
    layers.MaxPooling2D((2, 2), name='oculta2_pooling'),                       # → 2×2×6
    layers.Flatten(name='aplanado'),                                           # → 24 valores
    layers.Dense(6, activation='relu', name='oculta3_densa'),                  # capa totalmente conectada
    layers.Dense(3, activation='softmax', name='salida_softmax'),              # probabilidades X / O / +
], name='CNN_profunda')

modelo.compile(optimizer=keras.optimizers.Adam(0.01),
               loss='sparse_categorical_crossentropy',      # entropía cruzada
               metrics=['accuracy'])
modelo.summary()

In [ ]:
# Diagrama automático de Keras (en Colab ya están instalados pydot y graphviz)
try:
    from IPython.display import Image, display
    keras.utils.plot_model(modelo, to_file='modelo_keras.png', show_shapes=True, show_layer_names=True, dpi=80)
    display(Image('modelo_keras.png'))
except Exception as e:
    print('plot_model no disponible en este entorno:', e)

In [ ]:
# Propagación hacia adelante + retropropagación + descenso del gradiente, época tras época
historial = modelo.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)

perdida, exactitud = modelo.evaluate(X_test, y_test, verbose=0)
print(f'Exactitud en datos de prueba: {exactitud:.2%}   Pérdida: {perdida:.4f}')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(historial.history['loss'], label='Entrenamiento'); a1.plot(historial.history['val_loss'], label='Validación')
a1.set_title('Pérdida (entropía cruzada)'); a1.set_xlabel('Época'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(historial.history['accuracy'], label='Entrenamiento'); a2.plot(historial.history['val_accuracy'], label='Validación')
a2.set_title('Exactitud'); a2.set_xlabel('Época'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Figura 3 – La red con las predicciones reales del modelo

Se pasa la imagen de la **X** (la misma de la Figura 2) por el modelo entrenado y se dibuja de nuevo la Figura 1 con las probabilidades Softmax que devuelve.

In [ ]:
probs = modelo.predict(X_img[np.newaxis, ..., np.newaxis].astype('float32'), verbose=0)[0]
for c, p in zip(CLASES, probs):
    print(f'P({c}) = {p:.4f}')
print('Predicción:', CLASES[int(np.argmax(probs))])

# Los 4 píxeles de entrada del dibujo son el mapa 2×2 tras convolución + ReLU + pooling (Figura 2)
dibujar_red(clases=['“X”', '“O”', '“+”'], probs=probs, pixeles=pool_1.ravel(),
            titulo='Figura 3. Red profunda con las probabilidades predichas por el modelo entrenado',
            archivo='figura3_red_con_predicciones.png')

In [ ]:
# Descargar las figuras en alta resolución (solo funciona en Google Colab)
try:
    from google.colab import files
    for f in ['figura1_red_profunda.png', 'figura2_etapa_convolucional.png', 'figura3_red_con_predicciones.png']:
        files.download(f)
except ImportError:
    print('Fuera de Colab: las figuras quedaron guardadas en la carpeta actual.')

## Conclusiones

La profundidad de la red no es solo cantidad de neuronas: cada capa oculta transforma lo que recibe en una representación más abstracta. En la práctica, la capa convolucional aprende filtros parecidos a los de la Figura 2 (diagonales, bordes), el *pooling* resume esas respuestas y la capa densa las combina para decidir la clase. Los pesos *w* y sesgos *b* se ajustan con retropropagación y descenso del gradiente a partir de datos etiquetados, que es la esencia del aprendizaje supervisado (Goodfellow et al., 2016; LeCun et al., 2015).

## Referencias

Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep learning*. MIT Press. https://www.deeplearningbook.org/

LeCun, Y., Bengio, Y., & Hinton, G. (2015). Deep learning. *Nature, 521*(7553), 436–444. https://doi.org/10.1038/nature14539

LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE, 86*(11), 2278–2324. https://doi.org/10.1109/5.726791

Serrano.Academy. (2017a). *A friendly introduction to convolutional neural networks and image recognition* [Video]. YouTube. https://www.youtube.com/watch?v=2-Ol7ZB0MmU

Serrano.Academy. (2017b). *A friendly introduction to deep learning and neural networks* [Video]. YouTube. https://www.youtube.com/watch?v=BR9h47Jtqyw